# 🚀 GIAI ĐOẠN 5 — KIẾN TRÚC STAIR5-v5.1 / NLGCL-KPE: KNOWN-POSITIVE EXCLUSION TRÊN BACKBONE NLGCL-CSE
### 🏆 Kaggle ML Engineering Pipeline — Multimodal Recommendation with Known-Positive Exclusion Contrastive Learning & Contraction-Preserving Adaptive Mixing

> **Đề tài:** Recommender Systems using Graph Representation: Multi-modal  
> **Khóa luận tốt nghiệp:** Khóa 2021–2025 — Khoa Công nghệ Thông tin, Trường Đại học Khoa học Tự nhiên, ĐHQG-HCM  
> **Sinh viên thực hiện:** Lê Hà Thanh Chương (23120195) & Bùi Trung Hiếu (23120257)  
> **Giảng viên hướng dẫn:** TS. Nguyễn Ngọc Thảo  
> **Kiến trúc đề xuất:** **STAIR5-v5.1 / NLGCL-KPE** (*Neighborhood-enriched Contrastive Learning with Known-Positive Exclusion*)
> **Tài liệu phương pháp luận & Thiết kế:** `docs/giai_doan_5/STAIR5_v5.1_Report.md`  
> **Mã nguồn lõi:** `models/stair5_v51.py`, `models/stair5_v51_objectives.py`, `models/stair5_v51_graph.py`, `models/stair5_v51_utils.py`, `optimizers/stair5_v51_smoother.py`, `main_stair5_v51.py`  
> **Tập dữ liệu mục tiêu:** **Amazon Baby**, **Amazon Sports**, **Amazon Electronics** (3 tập benchmark chuẩn)  
> **Chiến lược thực thi:** **3-Tier Stage Execution** (`Stage A: Tiny/Smoke ➔ Stage B: Baby Pilot ➔ Stage C: Sports & Baby Full 500 ➔ Stage D: Electronics Scale-up`)
> **Đo lường VRAM:** Paper Standard (`peak_allocated_gib` & `vram_allocated_mb` trích xuất từ `training_telemetry.jsonl` và NVML)

---

## 📑 1. BẢN THIẾT KẾ KIẾN TRÚC ĐỀ XUẤT: STAIR5-v5.1 / NLGCL-KPE

| Thành Phần Trụ Cột | Cơ Chế Kỹ Thuật STAIR5-v5.1 | Ý Nghĩa Khoa Học & Đột Phá So Với v4 |
|:---|:---|:---|
| **1. Kế thừa Backbone v4** | MI SVD Whitening, Tích chập phổ FSC, toán tử BSC Smoother trong AdamWSEvo, hàm mất mát BPR, scorer dot-product Euclidean $O(1)$. | Giữ nguyên vẹn độ trễ suy luận, tái lập chính xác v4 khi $\rho=0$ hoặc arm `V4`. |
| **2. Unique Entity Keys** | Deduplicate key entity IDs ($K_U, K_I$) trong batch, chuyển GEMM từ $B \times B$ sang $B \times m$. | Loại bỏ lãng phí tính toán trên các key trùng lặp, chuẩn hóa xác suất InfoNCE trên không gian thực thể duy nhất. |
| **3. Giữ Nguyên B Query Occurrences** | Không deduplicate query rows; giữ nguyên $B$ sampled query occurrences với trọng số công bằng. | Tránh hiện tượng méo phân phối bậc và không thiên lệch sang các entity phổ biến khi tính toán gradient. |
| **4. Known-Positive Exclusion (KPE)** | $A_{bk} = \mathbf{1}[k = t_b] \lor \neg M_{bk}$; loại triệt để các train-positive khác khỏi mẫu số InfoNCE của từng query. | Triệt tiêu xung đột gradient giả (false-negative repulsion), bảo vệ các mối quan hệ tích cực đã biết trong tập huấn luyện. |
| **5. GPU Membership Searchsorted** | Tra cứu quan hệ train-positive bằng `torch.searchsorted` trên tensor 1D int64 pair keys $(u \cdot n_I + i)$ trực tiếp trên GPU. | Hoàn toàn không dùng vòng lặp Python, không gọi `.item()` hay tạo ma trận dense $U \times I$. |
| **6. Checkpoint Anchor Chunking** | Tái tính toán logits và membership trong từng chunk $(C \times m)$ khi backward (non-reentrant checkpoint). | Giữ peak VRAM ở mức cực thấp, bảo đảm an toàn bộ nhớ tuyệt đối trên tập dữ liệu lớn Electronics ($B=4096$). |
| **7. Toàn Tử CAM (Factorial Arm)** | $S_{\text{CAM}} = A S_0 A + B \bar{S}_{\text{CF}} B$ với $A=\text{diag}\sqrt{1-\eta}, B=\text{diag}\sqrt{\eta}$. | Bảo toàn chặn phổ $\|S_{\text{CAM}}\|_2 \le 1.0$ cho mọi $\delta$; khôi phục $S_4$ chính xác khi $\delta=0$. |


## Cell 1 ⚙️ Thiết lập Môi trường, Dependencies & Đồng bộ Mã Nguồn STAIR5-v5.1
- Đồng bộ mã nguồn mới nhất từ GitHub repository `ThanhChuong12/STAIR-Enhanced` (branch `main`).
- Cài đặt các gói phụ thuộc chuẩn tắc tương thích với môi trường nghiệm thu.
- Khởi chạy tiến trình con xác thực (`Subprocess Environment Verification`) để kiểm tra toàn bộ imports và GPU runtime.


In [ ]:
# Cell 1: Môi trường, Dependencies & Đồng bộ STAIR-Enhanced (STAIR5-v5.1 NLGCL-KPE)
import os, shutil, subprocess, sys, types, glob, torch
from pathlib import Path

STAIR_DIR = '/kaggle/working/STAIR-Enhanced'
os.chdir('/kaggle/working') if os.path.exists('/kaggle/working') else None

# 1. Đồng bộ repository STAIR-Enhanced thông minh & an toàn
if os.path.exists(STAIR_DIR):
    print('Thư mục STAIR-Enhanced đã tồn tại. Đang kiểm tra đồng bộ mã nguồn...')
    try:
        subprocess.run(['git', '-C', STAIR_DIR, 'fetch', 'origin', 'main'], check=True, capture_output=True)
        chk = subprocess.run(['git', '-C', STAIR_DIR, 'ls-tree', 'origin/main', 'models/stair5_v51.py'], capture_output=True, text=True)
        if 'models/stair5_v51.py' in chk.stdout:
            subprocess.run(['git', '-C', STAIR_DIR, 'reset', '--hard', 'origin/main'], check=True)
            print('✅ Đã đồng bộ về commit mới nhất của origin/main (chứa STAIR5-v5.1 NLGCL-KPE).')
        else:
            print('⚠️ Remote origin/main chưa có models/stair5_v51.py. Giữ nguyên mã nguồn hiện có trên đĩa.')
    except Exception as e:
        print(f'⚠️ Cảnh báo git fetch/check ({e}). Tiếp tục dùng mã nguồn hiện có trên đĩa.')

if not os.path.exists(STAIR_DIR) and os.path.exists('/kaggle/working'):
    print('Cloning STAIR-Enhanced repository (branch main)...')
    try:
        subprocess.run([
            'git', 'clone', '--depth', '1',
            'https://github.com/ThanhChuong12/STAIR-Enhanced.git', STAIR_DIR
        ], check=True)
    except Exception as e:
        print(f'⚠️ Git clone gặp cảnh báo: {e}')

# Fallback: Tự động dò quét và sao chép mã nguồn từ Kaggle Input nếu chưa có models/stair5_v51.py
if not os.path.exists(os.path.join(STAIR_DIR, 'models', 'stair5_v51.py')):
    input_matches = glob.glob('/kaggle/input/**/models/stair5_v51.py', recursive=True)
    if input_matches:
        src_repo = os.path.dirname(os.path.dirname(input_matches[0]))
        print(f'📦 Phát hiện mã nguồn STAIR5-v5.1 từ Kaggle Input: {src_repo} -> Sao chép vào {STAIR_DIR}...')
        os.makedirs(STAIR_DIR, exist_ok=True)
        shutil.copytree(src_repo, STAIR_DIR, dirs_exist_ok=True)

active_dir = STAIR_DIR if os.path.exists(STAIR_DIR) else os.path.abspath('.')
for p in [active_dir, STAIR_DIR, '/kaggle/working', '.']:
    if p and os.path.exists(p) and p not in sys.path:
        sys.path.insert(0, p)

if os.path.exists(STAIR_DIR):
    os.chdir(STAIR_DIR)

print(f'Active working directory: {os.getcwd()}')
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU Device: {torch.cuda.get_device_name(0)}')


## Cell 2 🧪 Kiểm định Đơn vị (Unit Tests & Correctness Suite)
- Chạy kiểm định toàn diện 9 test cases cho objectives (KPE, DEDUP, MP-MATCH, REF), CAM contraction bound $\le 1.0$, và pipeline/RNG roundtrip.


In [ ]:
# Cell 2: Thực thi bộ kiểm định Unit Tests tự động
import subprocess, sys
res = subprocess.run([sys.executable, '-m', 'pytest', 'tests/test_stair5_v51_objectives.py', 'tests/test_stair5_v51_graph.py', 'tests/test_stair5_v51_pipeline.py', '-v'], capture_output=True, text=True)
print(res.stdout)
if res.returncode != 0:
    print(res.stderr)
    raise RuntimeError('Unit tests failed! Please fix before proceeding.')
else:
    print('✅ Tất cả 9/9 unit tests đã VƯỢT QUA thành công!')


## Cell 3 🏃‍♂️ Huấn luyện & Đánh giá STAIR5-v5.1 trên Các Tập Dữ Liệu Benchmark
- Hỗ trợ chạy các tập dữ liệu: `Amazon Baby`, `Amazon Sports`, `Amazon Electronics`.
- Hỗ trợ các arms: `V51-KPE` (primary đề xuất), `V4` (comparator control), `DEDUP`, `MP-MATCH`, `CAM-KPE`.
- Tự động ghi nhật ký `training_telemetry.jsonl` với chuẩn đo lường kép: `peak_allocated_gib` và `vram_allocated_mb`.


In [ ]:
# Cell 3: Runner hàm thực thi huấn luyện STAIR5-v5.1
import os, subprocess, sys, time, json
from pathlib import Path

def run_stair5_v51(dataset_key='baby', arm='V51-KPE', epochs=500, seed=1, extra_args=None):
    config_map = {
        'baby': 'configs/Amazon2014Baby_STAIR5_v51.yaml',
        'sports': 'configs/Amazon2014Sports_STAIR5_v51.yaml',
        'electronics': 'configs/Amazon2014Electronics_STAIR5_v51.yaml',
    }
    cfg_file = config_map.get(dataset_key.lower())
    if not cfg_file or not os.path.exists(cfg_file):
        raise FileNotFoundError(f'Config file not found for dataset: {dataset_key}')
    
    artifact_dir = f'logs/GD5/stair5_v51/{dataset_key}_{arm}_seed{seed}'
    Path(artifact_dir).mkdir(parents=True, exist_ok=True)
    
    cmd = [
        sys.executable, 'main_stair5_v51.py',
        '--config', cfg_file,
        '--v51-arm', arm,
        '--epochs', str(epochs),
        '--seed', str(seed),
        '--artifact-dir', artifact_dir,
    ]
    if extra_args:
        cmd.extend(extra_args)
        
    print(f'🚀 Bắt đầu huấn luyện STAIR5-v5.1 [{dataset_key.upper()} | Arm: {arm} | Epochs: {epochs} | Seed: {seed}]...')
    print('Command:', ' '.join(cmd))
    started = time.perf_counter()
    res = subprocess.run(cmd)
    elapsed = time.perf_counter() - started
    print(f'🏁 Hoàn thành sau {elapsed/60:.2f} phút! Artifacts lưu tại: {artifact_dir}')
    return artifact_dir

# Ví dụ chạy Tiny/Smoke 2 epochs trên Baby để kiểm tra pipeline:
# run_stair5_v51(dataset_key='baby', arm='V51-KPE', epochs=2, extra_args=['--eval-freq', '1'])

# Chạy chính thức Amazon Baby 500 epochs:
artifact_baby = run_stair5_v51(dataset_key='baby', arm='V51-KPE', epochs=500, seed=1)


## Cell 4 📊 Trực quan hóa Telemetry, Quỹ đạo NDCG@20 & Hồ sơ Bộ nhớ VRAM Thực tế
- Trích xuất chuẩn xác từ `training_telemetry.jsonl` bằng cả hai khóa `peak_allocated_gib` (nhân 1024) và `vram_allocated_mb`.
- Vẽ đường cong học tập (BPR Loss, CL Loss, Validation NDCG@20).
- Vẽ biểu đồ tiêu thụ VRAM thực tế.


In [ ]:
# Cell 4: Trực quan hóa Telemetry và Phân tích Hội tụ
import json, os
from pathlib import Path
import matplotlib.pyplot as plt

def plot_v51_telemetry(artifact_dir, dataset_name='Amazon Baby'):
    p = Path(artifact_dir) / 'training_telemetry.jsonl'
    if not p.is_file():
        print(f'Chưa có file telemetry tại {p}')
        return
    
    records = []
    with open(p, 'r', encoding='utf-8') as f:
        for line in f:
            if line.strip():
                try:
                    records.append(json.loads(line))
                except Exception:
                    pass
                    
    if not records:
        print('Telemetry rỗng.')
        return
        
    epochs = [r['epoch'] for r in records]
    loss = [r['loss'] for r in records]
    bpr = [r.get('bpr_loss', 0.0) for r in records]
    cl = [r.get('nlgcl_loss', 0.0) for r in records]
    vram = [r.get('vram_allocated_mb', r.get('peak_allocated_gib', 0.0) * 1024) for r in records]
    
    fig, axs = plt.subplots(1, 2, figsize=(14, 5))
    
    # Loss trajectory
    axs[0].plot(epochs, loss, label='Total Loss', color='#1f77b4', lw=1.5)
    axs[0].plot(epochs, bpr, label='BPR Loss', color='#2ca02c', lw=1.5, ls='--')
    axs[0].plot(epochs, cl, label='KPE CL Loss', color='#d62728', lw=1.2, ls=':')
    axs[0].set_title(f'{dataset_name} — Động lực học tổn thất (Loss Dynamics)')
    axs[0].set_xlabel('Epoch')
    axs[0].set_ylabel('Loss')
    axs[0].grid(True, alpha=0.3)
    axs[0].legend()
    
    # VRAM profile
    axs[1].plot(epochs, vram, label='VRAM Peak (MiB)', color='#ff7f0e', lw=2)
    axs[1].set_title(f'{dataset_name} — Hồ sơ tiêu thụ VRAM thực tế')
    axs[1].set_xlabel('Epoch')
    axs[1].set_ylabel('VRAM Allocated (MiB)')
    axs[1].grid(True, alpha=0.3)
    axs[1].legend()
    
    plt.tight_layout()
    out_img = Path(artifact_dir) / 'telemetry_visualization.png'
    plt.savefig(out_img, dpi=300)
    plt.show()
    print(f'✅ Đã lưu đồ thị telemetry tại: {out_img}')

# plot_v51_telemetry(artifact_baby, 'Amazon Baby')
